# 07 — Selective recovery and replacement

**NOT RUN**

Prepared but intentionally not executed. Run manually in Google Colab.

Experiments E04/E07. CPU only; optional budget is configured in config.yaml.

Upload the project ZIP when prompted. Set only the flags for the stage you intend to run. Model stages need `INSTALL_MODEL_PACKAGES=True`; core stages need only PyYAML. Back up datasets/checkpoints/results to Drive manually before the Colab session ends.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import zipfile

PROJECT_DIR = Path.cwd() / "certificate-recovery"
ARCHIVE = Path.cwd() / "Dependency_Certificate_Recovery_Foundation.zip"
if not PROJECT_DIR.exists():
    if not ARCHIVE.exists():
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP")
        ARCHIVE = Path.cwd() / archives[0]
    with zipfile.ZipFile(ARCHIVE) as bundle:
        destination = Path.cwd().resolve()
        for member in bundle.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)
if not (PROJECT_DIR / "src" / "cert_recovery").exists():
    raise FileNotFoundError("Project source directory missing")
sys.path.insert(0, str(PROJECT_DIR / "src"))

INSTALL_CORE_PACKAGES = False
INSTALL_MODEL_PACKAGES = False
if INSTALL_CORE_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")])
if INSTALL_MODEL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-colab.txt")])
if importlib.util.find_spec("yaml") is None:
    raise RuntimeError("Set INSTALL_CORE_PACKAGES=True and rerun setup; no model packages are needed for the core.")

from cert_recovery.config import load_config
config = load_config(PROJECT_DIR / "config" / "config.yaml")
print("Prepared but intentionally not executed. Run manually in Google Colab.")


In [ ]:
from cert_recovery.config import resolve_path
from cert_recovery.data_pipeline import read_jsonl
from cert_recovery.evaluation import run_workflow_case, run_replacement_experiment

RUN_RECOVERY_EXPERIMENTS = False
if RUN_RECOVERY_EXPERIMENTS:
    config["execution"]["allow_research_experiments"] = True
    cases = read_jsonl(resolve_path(config, "workflow_cases"))
    recovery_records = [run_workflow_case(case, "proposed", config, manual=True) for case in cases]
    replacement = run_replacement_experiment(config, manual=True)
    print({"recovery": recovery_records, "replacement": replacement})
else:
    print("NOT RUN: recovery experiments")


Inspect blocked work, failed attempts and unaffected branches. Cheap abstention is not useful recovery. A removed source needs a declared trusted replacement; recovery does not invent one.